# 02 — Generazione FCGR Baseline, Analisi Sparsità/Memoria e Visualizzazione

## Scopo del Notebook
1. Testare l'algoritmo **FCGR Baseline** (`src.fcgr.compute_fcgr`) su uno studio pilota bilanciato tra le 2 classi (`0` = Stabile/Healthy, `1` = Hotspot) e le 3 fasce di lunghezza (`short < 300 bp`, `medium 300-1000 bp`, `long > 1000 bp`).
2. Misurare al variare di $k \in [4, 10]$:
   - **Sparsità ($S\%$)** e **Occupazione ($\%$)**;
   - **Tempo di calcolo medio (`ms`/sequenza)**;
   - **Memory Footprint (`MB`/`GB`)** per singolo campione, per il Training Set e per l'intero Manifest.
3. Visualizzare le matrici FCGR tramite la **Dashboard Diagnostica a 4 Pannelli** (`plot_fcgr_diagnostics`) per osservare direttamente cosa entra nella Rete Siamese.

In [ ]:
import time
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from pathlib import Path
import sys

# Rilevamento automatico Google Colab vs Ambiente Locale
COLAB_ROOT = Path('/content/drive/MyDrive/eccdna_project')
if COLAB_ROOT.exists():
    PROJECT_ROOT = COLAB_ROOT
else:
    CURRENT_DIR = Path.cwd().resolve()
    PROJECT_ROOT = CURRENT_DIR.parent if CURRENT_DIR.name == 'notebooks' else CURRENT_DIR

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

RAW_DIR = PROJECT_ROOT / 'data' / 'raw'
PROCESSED_DIR = PROJECT_ROOT / 'data' / 'processed'
RESULTS_DIR = PROJECT_ROOT / 'results' / 'fcgr_analysis'
ARTIFACTS_DIR = PROJECT_ROOT / 'artifacts'

PROCESSED_DIR.mkdir(parents=True, exist_ok=True)
RESULTS_DIR.mkdir(parents=True, exist_ok=True)
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

META_PATH = RAW_DIR / 'eccdna_disease_detection_metadata.tsv'
FASTA_PATH = RAW_DIR / 'eccdna_disease_detection.body.fa'
MANIFEST_PATH = PROCESSED_DIR / 'siamese_primary_manifest.tsv'

print('PROJECT_ROOT :', PROJECT_ROOT)
print('META_PATH    :', META_PATH, '-> exists:', META_PATH.exists())
print('FASTA_PATH   :', FASTA_PATH, '-> exists:', FASTA_PATH.exists())

from src.fcgr import (
    load_fasta_sequences,
    compute_fcgr,
    normalize_fcgr,
    compute_fcgr_diagnostics,
    plot_fcgr_diagnostics,
)

## 1. Caricamento del Manifest Binario e Campionamento Pilot Stratificato
Estraiamo dal training set un campione stratificato per **classe binaria (`0`, `1`)** e **fascia di lunghezza (`short`, `medium`, `long`)** e carichiamo le rispettive sequenze dal file FASTA.

In [ ]:
metadata = pd.read_csv(MANIFEST_PATH, sep='\t', low_memory=False)
metadata['id'] = metadata['id'].astype(str)

if 'length_bin' not in metadata.columns:
    metadata['length_bin'] = np.where(metadata['length'] < 300, 'short', np.where(metadata['length'] <= 1000, 'medium', 'long'))

train_metadata = metadata[metadata['split_cluster'] == 'train'].copy()

RANDOM_STATE = 42
N_PER_STRATUM = 150  # 2 classi x 3 fasce di lunghezza x 150 = 900 sequenze nel pilot

pilot_parts = []
for cid in [0, 1]:
    for b_idx, lbin in enumerate(['short', 'medium', 'long']):
        sub = train_metadata[(train_metadata['class_id'] == cid) & (train_metadata['length_bin'] == lbin)]
        if len(sub) > 0:
            sampled = sub.sample(n=min(N_PER_STRATUM, len(sub)), replace=False, random_state=RANDOM_STATE + cid * 10 + b_idx)
            pilot_parts.append(sampled)

pilot_df = pd.concat(pilot_parts, ignore_index=True)
print(f'Sequenze selezionate per il pilot: {len(pilot_df)}')
display(pd.crosstab(pilot_df['disease_clean'], pilot_df['length_bin']))

t0 = time.perf_counter()
pilot_sequences = load_fasta_sequences(FASTA_PATH, pilot_df['id'].tolist())
print(f'Sequenze caricate dal FASTA: {len(pilot_sequences)} in {time.perf_counter() - t0:.2f} s')
assert len(pilot_sequences) == len(pilot_df), 'Alcuni ID del pilot non sono stati trovati nel FASTA!'

## 2. Dashboard Diagnostica Visiva della FCGR (`plot_fcgr_diagnostics`)
Ispezioniamo visivamente la FCGR ($k=6$, `64x64`) di una sequenza **Stabile (`class_id=0`)** e di un **Hotspot (`class_id=1`)**, confrontando anche una sequenza corta (`short`) con una lunga (`long`).

In [ ]:
# Visualizzazione su un esempio di classe 0 (Stabile/Healthy) e uno di classe 1 (Hotspot)
for cid in [0, 1]:
    row = pilot_df[pilot_df['class_id'] == cid].iloc[0]
    seq = pilot_sequences[row['id']]
    _ = plot_fcgr_diagnostics(
        sequence_or_fcgr=seq,
        k=6,
        sample_id=row['id'],
        disease=f"{row['disease_clean']} (label={cid}, {row['length_bin']})",
        save_path=RESULTS_DIR / f'fcgr_diagnostic_class_{cid}.png',
    )

## 3. Benchmark Sistematico della FCGR Baseline per $k \in [4, 10]$
Misuriamo per ogni sequenza del pilot e per ogni $k \in [4, 10]$ la **sparsità**, l'**unique ratio**, il **tempo di conversione (`ms`)** e il **consumo di memoria (`MB`/`GB`)**.

In [ ]:
K_VALUES = list(range(4, 11))
fcgr_records = []

for _, row in pilot_df.iterrows():
    seq_id = row['id']
    seq = pilot_sequences[seq_id]
    for k in K_VALUES:
        t_start = time.perf_counter()
        counts, n_valid, n_distinct = compute_fcgr(seq, k, return_distinct=True)
        _ = normalize_fcgr(counts)
        elapsed_ms = (time.perf_counter() - t_start) * 1000.0

        total_cells = counts.size
        occupancy = (n_distinct / total_cells) * 100.0
        sparsity = 100.0 - occupancy
        unique_ratio = (n_distinct / n_valid) if n_valid > 0 else np.nan

        fcgr_records.append({
            'id': seq_id,
            'class_id': int(row['class_id']),
            'label_name': row['disease_clean'],
            'length_bin': row['length_bin'],
            'sequence_length': len(seq),
            'k': k,
            'fcgr_size': 2 ** k,
            'total_cells': total_cells,
            'valid_kmers': n_valid,
            'distinct_kmers': n_distinct,
            'unique_ratio': unique_ratio,
            'occupancy_%': occupancy,
            'sparsity_%': sparsity,
            'time_ms': elapsed_ms,
        })

fcgr_results_df = pd.DataFrame(fcgr_records)
print('Benchmark completato. Totale valutazioni:', len(fcgr_results_df))

In [ ]:
n_train_samples = int((metadata['split_cluster'] == 'train').sum())
n_manifest_samples = len(metadata)

summary_by_k = (
    fcgr_results_df.groupby('k')
    .agg(
        n_sequences=('id', 'count'),
        median_length=('sequence_length', 'median'),
        median_valid_kmers=('valid_kmers', 'median'),
        median_distinct_kmers=('distinct_kmers', 'median'),
        median_unique_ratio=('unique_ratio', 'median'),
        median_occupancy=('occupancy_%', 'median'),
        q25_occupancy=('occupancy_%', lambda x: x.quantile(0.25)),
        q75_occupancy=('occupancy_%', lambda x: x.quantile(0.75)),
        median_sparsity=('sparsity_%', 'median'),
        mean_time_ms=('time_ms', 'mean'),
        median_time_ms=('time_ms', 'median'),
    )
    .reset_index()
)

memory_rows = []
for k in K_VALUES:
    side = 2 ** k
    cells = side * side
    bytes_f32 = cells * 4
    memory_rows.append({
        'k': k,
        'fcgr_shape': f'{side}x{side}',
        'cells': cells,
        'memory_float32_MB': bytes_f32 / (1024 ** 2),
        'train_dense_float32_GB': (bytes_f32 * n_train_samples) / (1024 ** 3),
        'full_manifest_dense_float32_GB': (bytes_f32 * n_manifest_samples) / (1024 ** 3),
    })
memory_by_k = pd.DataFrame(memory_rows)

final_k_summary = summary_by_k.merge(memory_by_k, on='k', how='left')
summary_by_length_k = (
    fcgr_results_df.groupby(['length_bin', 'k'])
    .agg(
        n=('id', 'count'),
        median_length=('sequence_length', 'median'),
        median_distinct=('distinct_kmers', 'median'),
        median_unique_ratio=('unique_ratio', 'median'),
        median_occupancy=('occupancy_%', 'median'),
        median_sparsity=('sparsity_%', 'median'),
        mean_time_ms=('time_ms', 'mean'),
    )
    .reset_index()
)

# Salvataggio CSV dei risultati baseline
summary_by_k.to_csv(RESULTS_DIR / 'summary_by_k.csv', index=False)
summary_by_length_k.to_csv(RESULTS_DIR / 'summary_by_length_k.csv', index=False)
memory_by_k.to_csv(RESULTS_DIR / 'memory_by_k.csv', index=False)
final_k_summary.to_csv(RESULTS_DIR / 'final_k_summary.csv', index=False)
fcgr_results_df.to_csv(RESULTS_DIR / 'fcgr_pilot_results.csv', index=False)

print('=== TABELLA RIEPILOGATIVA BASELINE FCGR ===')
display(final_k_summary[['k', 'fcgr_shape', 'median_distinct_kmers', 'median_sparsity', 'mean_time_ms', 'memory_float32_MB', 'train_dense_float32_GB']])

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))

for lbin in ['short', 'medium', 'long']:
    sub = summary_by_length_k[summary_by_length_k['length_bin'] == lbin]
    axes[0].plot(sub['k'], sub['median_sparsity'], marker='o', label=lbin)
axes[0].set_xlabel('k')
axes[0].set_ylabel('Sparsità Mediana (%)')
axes[0].set_title('Sparsità FCGR per Fascia di Lunghezza')
axes[0].grid(alpha=0.3)
axes[0].legend()

axes[1].plot(final_k_summary['k'], final_k_summary['mean_time_ms'], marker='s', color='#D69E2E')
axes[1].set_xlabel('k')
axes[1].set_ylabel('Tempo Medio per Sequenza (ms)')
axes[1].set_title('Tempo di Calcolo Baseline (Python Loop)')
axes[1].grid(alpha=0.3)

axes[2].semilogy(final_k_summary['k'], final_k_summary['train_dense_float32_GB'], marker='^', color='#C53030', label='Train Set')
axes[2].set_xlabel('k')
axes[2].set_ylabel('RAM / Disco float32 (GB)')
axes[2].set_title('Esplosione Memoria Densa (float32)')
axes[2].grid(alpha=0.3)
axes[2].legend()

plt.tight_layout()
plt.savefig(RESULTS_DIR / 'fcgr_baseline_scalability.png', dpi=300, bbox_inches='tight')
plt.show()